# Sesión 5 · Grandes modelos de lenguaje

**Módulo 3 · Modelos y APIs** · 2 sesiones · LLMs · prompt engineering · integración

> Entramos en el terreno de los modelos de lenguaje (los que hay detrás de Claude, ChatGPT o Gemini). Hoy es una sesión sobre todo **conceptual y de práctica con prompts**: entenderás qué son, cómo funcionan por dentro y cómo hablarles bien. El código para llamarlos lo montaremos en la Sesión 6.

**Al terminar la sesión sabrás:**
- Explicar qué es un Gran Modelo de Lenguaje (LLM) y nombrar las familias principales
- Entender, sin matemáticas, cómo funciona por dentro
- Aplicar técnicas básicas de *prompt engineering* (el arte de pedir bien)
- Usar *few-shot*, *chain-of-thought* y *system prompts*

*Resultado de la sesión: sabes qué es un LLM y escribes prompts que funcionan.*

## 1. ¿Qué es un Gran Modelo de Lenguaje (LLM)?

Un **Gran Modelo de Lenguaje** (*Large Language Model*, LLM) es un modelo de IA entrenado con cantidades enormes de texto para hacer una tarea aparentemente simple: **predecir qué palabra viene después**. De esa habilidad, llevada al extremo, emerge la capacidad de conversar, resumir, traducir o programar.

Conecta con lo que ya sabes: un LLM es una **red neuronal profunda** (Sesión 2) gigantesca, construida sobre la arquitectura *Transformer* que apareció en 2017 (Sesión 1). Lo "grande" viene de su tamaño: miles de millones de esos *pesos* que ajusta durante el entrenamiento.

Las familias principales hoy son:
- **Claude** — desarrollado por Anthropic
- **GPT** — desarrollado por OpenAI
- **Gemini** — desarrollado por Google
- **Llama** — desarrollado por Meta, con la particularidad de ser de pesos abiertos (puedes descargarlo y ejecutarlo tú)

> Dato útil para el curso: usaremos **Claude** (Anthropic) como ejemplo principal y, en paralelo, **modelos gratuitos** (Llama a través de OpenRouter, y Llama en local con Ollama). Así ves que la idea es la misma sin importar el proveedor.

## 2. Cómo funciona un LLM por dentro (sin matemáticas)

Vamos por partes, con calma.

**Paso 1 · El texto se trocea en *tokens*.** Un modelo no ve "palabras" como nosotros, sino piezas llamadas **tokens**: trozos de palabra. Por ejemplo, "increíble" podría partirse en `incre` + `íble`. Como regla muy aproximada, en español **un token equivale más o menos a 3 o 4 letras**. Esto importa porque el coste y los límites de las APIs se miden en tokens.

**Paso 2 · Predice el siguiente token.** Dado un texto, el modelo calcula qué token es más probable que venga después, lo añade, y repite. Escribe **palabra a palabra**, como quien completa la frase "el cielo es de color..." con "azul".

**Paso 3 · El entrenamiento.** Antes de poder hacer eso, el modelo leyó cantidades inmensas de texto y fue ajustando sus pesos hasta predecir bien. Después se afina para que sea útil y siga instrucciones.

**La consecuencia clave que debes recordar:** un LLM **no consulta una base de datos de hechos**; genera la continuación más probable. Por eso a veces produce respuestas que suenan seguras pero son falsas. A ese fenómeno lo llamamos **alucinación**, y es la razón por la que conviene verificar lo importante.

### ¿Qué es la "ventana de contexto"?

El modelo solo "ve" una cantidad limitada de texto a la vez: su **ventana de contexto**, medida en tokens. Incluye tu pregunta más toda la conversación previa que le pases. Si te sales de esa ventana, el modelo "olvida" lo más antiguo.

Idea práctica: el modelo **no recuerda** conversaciones anteriores por sí solo. Si quieres que tenga memoria de lo dicho, tienes que **reenviarle el historial** en cada llamada. Lo veremos en código en la Sesión 6.

## 3. Prompt engineering: el arte de pedir bien

Un **prompt** es la instrucción que le das al modelo. *Prompt engineering* es, sencillamente, **aprender a pedir de forma que obtengas buenos resultados**. No hace falta ser técnico: es comunicación clara. Estas son las técnicas básicas que más rendimiento dan.

**Técnica 1 · Sé específico y da contexto.** Cuanto más claro el qué, el para qué y el para quién, mejor.

| ❌ Prompt vago | ✅ Prompt específico |
|---|---|
| "Escribe sobre digitalización" | "Escribe un párrafo de 5 líneas, en tono divulgativo, que explique a una pyme del sector agroalimentario por qué le conviene digitalizar su trazabilidad" |

**Técnica 2 · Indica el formato de salida.** Pide explícitamente la forma: "responde en una lista de 3 puntos", "devuelve solo un JSON con las claves nombre y precio", "máximo 50 palabras".

**Técnica 3 · Divide las tareas complejas.** Si la tarea es grande, pártela en pasos. Modelos y personas trabajamos mejor así.

### Técnica 4 · System prompt: asignar un rol

Casi todas las APIs separan dos tipos de mensaje:
- **System prompt** — instrucciones generales de comportamiento: quién es el modelo, su tono, sus reglas. Se fija una vez y condiciona toda la conversación
- **User prompt** — la petición concreta de cada turno

Ejemplo de *system prompt*:
> *"Eres un asistente experto en transformación digital para pymes. Respondes en español, con lenguaje sencillo, evitando tecnicismos innecesarios. Si no estás seguro de un dato, lo dices."*

Con eso, todas las respuestas saldrán con ese rol y ese tono sin tener que repetirlo cada vez.

### Técnica 5 · Few-shot: enseñar con ejemplos

*Few-shot* significa darle **algunos ejemplos resueltos** dentro del prompt para que imite el patrón. Es como mostrar dos o tres casos antes de pedirle el siguiente.

Ejemplo para clasificar el sentimiento de un comentario:

```
Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRO.

Comentario: "El producto llegó tarde y mal embalado"  ->  NEGATIVO
Comentario: "Funciona perfectamente, muy contento"     ->  POSITIVO
Comentario: "Es lo que esperaba, sin más"              ->  NEUTRO
Comentario: "La atención al cliente fue excelente"     ->
```

El modelo, al ver el patrón, completará con `POSITIVO`. Cuando das **cero** ejemplos se llama *zero-shot*; con **uno**, *one-shot*; con **varios**, *few-shot*.

### Técnica 6 · Chain-of-thought: pedir que razone paso a paso

Para problemas que requieren razonar (cálculos, lógica, decisiones con varios factores), pedirle al modelo que **piense paso a paso antes de responder** mejora mucho los resultados. Es la técnica *chain-of-thought* (cadena de pensamiento).

Basta con añadir algo como:
> *"Razona paso a paso antes de dar la respuesta final."*

En lugar de soltar un número que puede estar mal, el modelo desglosa el problema, y al "pensar en voz alta" acierta más. Es la misma idea que cuando a una persona le pides que muestre el desarrollo en vez de solo el resultado.

### 🟢 Ejercicio 1 · Reescribe un prompt vago

Tienes este prompt flojo:
> *"Háblame de la IA en la empresa"*

**Reescríbelo** aplicando las técnicas 1 y 2: añade contexto (público, objetivo) y especifica el formato de salida. Pruébalo en cualquier chatbot (Claude, ChatGPT...) y compara la respuesta con la del prompt original.

*No hay solución única: compártelo en clase y lo comentamos.*

### 🟢 Ejercicio 2 · Diseña un prompt few-shot

Quieres clasificar correos como `URGENTE` o `NO URGENTE`. Escribe un prompt few-shot con **3 ejemplos resueltos** y un cuarto correo sin resolver para que el modelo lo complete. Piensa qué ejemplos representan bien cada categoría.

## 4. Apéndice opcional · Tu primera llamada (anticipo de la Sesión 6)

Si quieres ir abriendo boca, esta celda muestra **cómo se vería** una llamada a Claude desde Python. **No la ejecutes todavía**: en la Sesión 6 instalamos todo y configuramos las claves con calma. Aquí solo es para que le pierdas el miedo: fíjate en que el *system prompt* y el *user prompt* son justo lo que acabamos de ver.

```python
import anthropic

cliente = anthropic.Anthropic()   # usa tu clave de la variable de entorno

respuesta = cliente.messages.create(
    model="claude-haiku-4-5",
    max_tokens=300,
    system="Eres un experto en transformación digital. Respondes en español y claro.",
    messages=[{"role": "user", "content": "Explica qué es un LLM en 3 frases."}],
)
print(respuesta.content[0].text)
```

Reconoces las piezas: el rol va en `system`, tu pregunta en `messages`. **Eso es todo.** Lo demás son detalles que dominaremos la próxima sesión.

## 5. Resumen de la sesión

- Un **LLM** es una red neuronal enorme que aprende a **predecir el siguiente token**; de ahí emerge su capacidad de lenguaje
- No consulta hechos: **genera lo más probable**, por eso puede **alucinar** y conviene verificar
- Solo ve su **ventana de contexto**; no recuerda por sí solo (hay que reenviarle el historial)
- *Prompt engineering* = pedir bien: **sé específico, marca el formato, asigna un rol** (*system prompt*), **da ejemplos** (*few-shot*) y **pide razonar paso a paso** (*chain-of-thought*)

**Para la próxima sesión:** lo llevamos a código. Llamarás a Claude y a modelos gratuitos desde Python, ajustarás parámetros y construirás tu primer chatbot.

👉 **Antes de la Sesión 6, crea una cuenta gratuita en Anthropic** (`console.anthropic.com`) **y otra en OpenRouter** (`openrouter.ai`) para tener tus claves listas. Si tu equipo es potente y quieres la opción 100% local, instala **Ollama** desde `ollama.com`.